# ex020_Splitting_RedoxFront2D

In [ ]:
from pathlib import Path

CASE_DIR = Path.cwd()
OUTPUT_DIR = CASE_DIR / "output"
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import Normalize, TwoSlopeNorm

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {"figure.dpi": 120, "savefig.dpi": 240, "font.size": 10.0, "axes.titleweight": "bold"}
)
with np.load(OUTPUT_DIR / "final_fields_comparison.npz") as archive:
    fields = {name: archive[name].copy() for name in archive.files}
metrics = pd.read_csv(OUTPUT_DIR / "comparison_metrics.csv")
labels = ("SNIA", "Strang", "SIA", "Reference")
comparison_methods = ("SNIA", "Strang", "SIA")
components = ("Don", "Extent")
coarse = metrics.set_index("label").loc[list(comparison_methods)].copy()
capacity = fields["solid_oxidant_capacity_model_mol"]
summary_columns = [
    "combined_nrmse",
    "Don_nrmse",
    "Extent_nrmse",
    "transport_solves",
    "reaction_evaluations",
    "wall_time_seconds",
    "oxidant_capacity_utilization_fraction",
    "donor_centroid_x_m",
]
domain_extent = tuple(fields["domain_extent_m"])
lens = fields["reactive_lens_mask"].astype(bool)
x = fields["x_cell_centers_m"]
y = fields["y_cell_centers_m"]
figure, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
context_specs = (
    ("hydraulic_conductivity_m_per_day", "viridis", "Hydraulic conductivity", "m day$^{-1}$"),
    (
        "solid_oxidant_capacity_model_mol",
        "magma",
        "Solid oxidant capacity",
        "model mol per chemistry cell",
    ),
)
for axis, (key, cmap, title, unit) in zip(axes, context_specs, strict=False):
    image = axis.imshow(fields[key], origin="lower", extent=domain_extent, aspect="auto", cmap=cmap)
    axis.contour(
        x, y, lens.astype(float), levels=[0.5], colors="white", linewidths=1.0, linestyles="--"
    )
    axis.set(title=title, xlabel="x (m)", ylabel="y (m)")
    figure.colorbar(image, ax=axis, shrink=0.86, label=unit)
figure.suptitle("Heterogeneous channel and ferric-oxide lens", fontsize=13)
plt.show()
display(coarse[summary_columns])

In [ ]:
donor_max = max(float(np.max(fields[f"{label}_Don"])) for label in labels)
row_specs = (
    ("Don", Normalize(0.0, max(donor_max, 1e-12)), "Blues", "Donor (mol L$^{-1}$)"),
    ("Extent", Normalize(0.0, 1.0), "Oranges", "Consumed solid capacity (fraction)"),
)
figure, axes = plt.subplots(
    2, 4, figsize=(11, 5), sharex=True, sharey=True, constrained_layout=True
)
mappables = []
for column, label in enumerate(labels):
    for row, (component, norm, cmap, colorbar_label) in enumerate(row_specs):
        values = fields[f"{label}_{component}"]
        if component == "Extent":
            values = values / capacity
        axis = axes[row, column]
        image = axis.imshow(
            values, origin="lower", extent=domain_extent, aspect="auto", cmap=cmap, norm=norm
        )
        axis.contour(
            x, y, lens.astype(float), levels=[0.5], colors="black", linewidths=0.75, linestyles="--"
        )
        if row == 0:
            axis.set_title(label)
        if column == 0:
            axis.set_ylabel(("Donor" if row == 0 else "Capacity used") + "\ny (m)")
        if row == 1:
            axis.set_xlabel("x (m)")
        if column == 0:
            mappables.append((image, colorbar_label))
for row, (image, colorbar_label) in enumerate(mappables):
    figure.colorbar(image, ax=list(axes[row, :]), shrink=0.72, label=colorbar_label)
figure.suptitle("Final capacity-limited redox front after pulse and flush", fontsize=13)
plt.show()

In [ ]:
differences = {
    (method, component): fields[f"{method}_{component}"] - fields[f"Reference_{component}"]
    for method in comparison_methods
    for component in components
}
limits = {
    component: max(
        float(np.max(np.abs(differences[method, component]))) for method in comparison_methods
    )
    for component in components
}
figure, axes = plt.subplots(
    2, 3, figsize=(10, 5), sharex=True, sharey=True, constrained_layout=True
)
for column, method in enumerate(comparison_methods):
    for row, component in enumerate(components):
        limit = max(limits[component], 1e-15)
        image = axes[row, column].imshow(
            differences[method, component],
            origin="lower",
            extent=domain_extent,
            aspect="auto",
            cmap="coolwarm",
            norm=TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit),
        )
        axes[row, column].contour(
            x, y, lens.astype(float), levels=[0.5], colors="black", linewidths=0.65, linestyles="--"
        )
        if row == 0:
            axes[row, column].set_title(method)
        if column == 0:
            axes[row, column].set_ylabel(
                ("$\\Delta$Donor" if row == 0 else "$\\Delta$oxidant use") + "\ny (m)"
            )
        if row == 1:
            axes[row, column].set_xlabel("x (m)")
        figure.colorbar(image, ax=axes[row, column], shrink=0.78)
figure.suptitle("Coarse method minus 0.125-day Strang reference", fontsize=13)
plt.show()

In [ ]:
colors = ["#4c78a8", "#f58518", "#54a24b"]
figure, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
axes[0].bar(coarse.index, coarse["combined_nrmse"], color=colors)
axes[0].set(title="Unified field error", ylabel="Combined NRMSE")
for index, value in enumerate(coarse["combined_nrmse"]):
    axes[0].text(index, value, f"{value:.4f}", ha="center", va="bottom", fontsize=9)
axes[1].bar(coarse.index, coarse["wall_time_seconds"], color=colors)
axes[1].set_yscale("log")
axes[1].set(title="Observed runtime", ylabel="Wall time (s, log scale)")
for index, value in enumerate(coarse["wall_time_seconds"]):
    axes[1].text(index, value, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
axes[2].scatter(
    coarse["transport_solves"],
    coarse["combined_nrmse"],
    s=90,
    c=colors,
    edgecolor="black",
    linewidth=0.5,
)
for label, row in coarse.iterrows():
    axes[2].annotate(
        label,
        (row["transport_solves"], row["combined_nrmse"]),
        xytext=(5, 5),
        textcoords="offset points",
    )
axes[2].set_xscale("log")
axes[2].set(
    xlabel="Transport solves (log scale)", ylabel="Combined NRMSE", title="Accuracy–work trade-off"
)
for axis in axes:
    axis.grid(True, alpha=0.25)
plt.show()
display(
    coarse[
        [
            "combined_nrmse",
            "Don_nrmse",
            "Extent_nrmse",
            "transport_solves",
            "reaction_evaluations",
            "total_sia_iterations",
            "wall_time_seconds",
            "oxidant_capacity_utilization_fraction",
            "ninety_percent_depleted_area_m2",
        ]
    ]
)